In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from polygon_fuggvenyek import *
from poligon_szk_fuggvenyek import *

In [2]:
def poly_gen_pipeline(VAROS, MAX_EXT=200.0, EPS=0.25, DIST_LIM=100.0, MIN_SEG=0.1):

    # 1. poligonok létrehozása
    
    # letöltöm a szükséges adatokat osm-ről ()
    Gp, nodes, edges, res_p, city_boundary = letoltes(VAROS)
    res_cut = vag_residential_city(res_p, city_boundary)
    res_area, boundary = res_area_es_boundary(res_cut, edges)
    
    orange = orange_gen(Gp, nodes, edges, MAX_EXT=MAX_EXT, EPS=EPS, MIN_SEG=MIN_SEG)
    blue = blue_gen(nodes, boundary, DIST_LIM=DIST_LIM, MIN_SEG=MIN_SEG)

    network_gs_proj = kapcsolas(edges, orange, blue, res_area)

    # itt vannak gdf-ben az összes generált polygon
    polygons_merged_gdf = egyesites(network_gs_proj)

    #polygons_merged_gdf.to_file('../../adatok/working/test5_v3.gpkg', layer='network_polygons', driver='GPKG')

    # következő lépéshez tudom kell a hivatalos városhatárt
    # ...

    gdf_szigetek = polygons_merged_gdf # ha jó használd az eredeti nevet
    return gdf_szigetek

In [5]:
def generalas_pipeline(VAROS, DATE):

    # beolvasom az összekapcsolt pontok df-et
    gdf = gpd.read_file('../../adatok/working/osszekapcsolt_pontok_v1.gpkg')

    # szűröm városra és választásra
    gdf = gdf.query('date == @DATE & telepulesnev_hu == @VAROS')

    # hozzárendelem a színeket a szavazókörökhöz (qgis vizualizációhoz)
    gdf = add_color_to_gdf(gdf)

    # legenerálom (később beolvasom) a beazonosítandó település parcellákat
    gdf_szigetek = poly_gen_pipeline(VAROS)

    # szavazókörhöz rendelem a poligonokat
    results = pontok_polygonban(gdf, gdf_szigetek, max_depth=45)

    # azokat a területeket amiben nincsen cím hozzárendelem a legnagyobb átfedésű szomszéd szavazókörhöz
    results_filled = ures_polyk_besorolasa(results)

    # a kis parcellákat egyesítem egyetelen multypolygonba
    merged = polygonok_egyesitese(results_filled, start_tol=0.2, max_tol=20)

    # export qgis-be
    merged.to_file(f'../../adatok/working/{VAROS}_szigetek_besorolt_14.gpkg', layer='network_polygons', driver='GPKG')

    return gdf, gdf_szigetek

In [7]:
VAROS = 'Szigetszentmiklós'
DATE = '2014-04-06'

gdf, gdf_szigetek = generalas_pipeline(VAROS, DATE)

Szavazókörök száma 30
Ellenőrzés: symmetric_difference area (terület eltérés): 0.0
Elérte a poly a mélységi szintet!
Elérte a poly a mélységi szintet!
Elérte a poly a mélységi szintet!


In [10]:
gdf

,szavazokorid,kozteruletid,kozteruletnevid,utca,cim,telepulesid,telepulesnev,telepulesnev_hu,eventfromid,date,gid,utca_gdf,cim_gdf,telepules,iszam,orszag,geometry,color
949814,294202,12096,411397,Felsőtag,106,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.740020e+13,Felsőtag,106,Szigetszentmiklós,2310.0,Hungary,POINT (19.0863 47.40022),#5d86ea
949815,294202,12096,411397,Felsőtag,108,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.739506e+13,Felsőtag,108,Szigetszentmiklós,2310.0,Hungary,POINT (19.09107 47.39505),#5d86ea
949816,294202,12096,411397,Felsőtag,110,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.740099e+13,Felsőtag,110,Szigetszentmiklós,2310.0,Hungary,POINT (19.08572 47.40101),#5d86ea
949817,294202,12096,411397,Felsőtag,112,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.740112e+13,Felsőtag,112,Szigetszentmiklós,2310.0,Hungary,POINT (19.08547 47.40111),#5d86ea
949818,294202,12096,411397,Felsőtag,118,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.739420e+13,Felsőtag,118,Szigetszentmiklós,2310.0,Hungary,POINT (19.0917 47.39416),#5d86ea
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953440,294231,1442142,1580874,Gyökér utca,1,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.731658e+13,Gyökér utca,1,Szigetszentmiklós,2310.0,Hungary,POINT (19.03406 47.31658),#51adcc
953441,294231,1444556,1583273,Berek utca,4,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.732407e+13,Berek utca,4,Szigetszentmiklós,2310.0,Hungary,POINT (19.03354 47.32404),#51adcc
953442,294231,1452800,1591428,Bóbitás köz,7,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.732014e+13,Bóbitás köz,7,Szigetszentmiklós,2310.0,Hungary,POINT (19.03671 47.32014),#51adcc
953443,294231,1459240,1597783,Topolyafa utca,1,2631,Nigglau,Szigetszentmiklós,3611,2014-04-06,4.732189e+13,Topolyafa utca,1,Szigetszentmiklós,2310.0,Hungary,POINT (19.03661 47.32189),#51adcc


In [52]:
gdf = gpd.read_file('../../adatok/working/osszekapcsolt_pontok_v1.gpkg')
gdf = gdf.query('date == "2022-04-03" & telepulesnev_hu == "Szigetszentmiklós"')
gdf = add_color_to_gdf(gdf)
gdf.to_file('../../adatok/working/Szigetszentmiklós_cimek_besorolt_jo.gpkg', layer='network_polygons', driver='GPKG')

In [2]:
gdf

NameError: name 'gdf' is not defined

In [56]:
gdf.to_file('../../adatok/working/Szigetszentmiklós_cimek_besorolt_jo.gpkg', layer='network_polygons', driver='GPKG')

In [1]:
gdf.query('telepules == "Szigetszetnmiklós"')

NameError: name 'gdf' is not defined